# Milestone 1: Data Exploration & Retrieval

This notebook covers:
1. Data download and exploration using DuckDB
2. Corpus preprocessing and parquet export
3. Building and testing BM25, semantic, and hybrid retrievers
4. Qualitative evaluation across a query set

In [11]:
import sys
import duckdb
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
INDICES_DIR = PROJECT_ROOT / "indices"

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
INDICES_DIR.mkdir(parents=True, exist_ok=True)

CATEGORY = "All_Beauty"
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
META_URL = f"{BASE_URL}/meta_categories/meta_{CATEGORY}.jsonl.gz"

con = duckdb.connect()

## 1. Data Download & Exploration

Using DuckDB to download and explore the All Beauty metadata directly from the McAuley Lab URL.

In [12]:
head_meta = con.execute(f"SELECT * FROM read_json_auto('{META_URL}') LIMIT 5").df()
head_meta

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,All Beauty,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,10,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Howard Products,[],{'Package Dimensions': '7.1 x 5.5 x 3 inches; ...,B01CUPMQZE,None
1,All Beauty,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,4.5,3,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Yes To,[],"{'Item Form': 'Powder', 'Skin Type': 'Acne Pro...",B076WQZGPM,None
2,All Beauty,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,26,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Levine Health Products,[],{'Manufacturer': 'Levine Health Products'},B000B658RI,None
3,All Beauty,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",3.1,102,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Cherioll,[],"{'Brand': 'Cherioll', 'Item Form': 'Powder', '...",B088FKY3VD,None
4,All Beauty,Precision Plunger Bars for Cartridge Grips – 9...,4.3,7,"[Material: 304 Stainless Steel; Brass tip, Len...",[The Precision Plunger Bars are designed to wo...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Precision,[],{'UPC': '644287689178'},B07NGFDN6G,None


In [13]:
# Count total products (reads from URL)
count_df = con.execute(f"SELECT COUNT(*) AS total FROM read_json_auto('{META_URL}')").df()
print(f"Total products: {count_df['total'][0]:,}")

Total products: 112,590


In [14]:
# Download full metadata and save as parquet locally
con.execute(f"""
    COPY (SELECT * FROM read_json_auto('{META_URL}'))
    TO '{DATA_RAW / "meta_All_Beauty.parquet"}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")
print(f"Saved to {DATA_RAW / 'meta_All_Beauty.parquet'}")

Saved to /Users/williamchong/Documents/UBC_MDS/Block6/575advml/DSCI_575_project_willchh_jiromig/data/raw/meta_All_Beauty.parquet


In [15]:
# Working from local parquet (fast)
meta_df = con.execute(f"""
    SELECT * FROM read_parquet('{DATA_RAW / "meta_All_Beauty.parquet"}')
""").df()

print(f"Shape: {meta_df.shape}")
print(f"\nColumns: {list(meta_df.columns)}")
print("\nNull counts:")
print(meta_df.isnull().sum())

Shape: (112590, 14)

Columns: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together']

Null counts:
main_category           0
title                   0
average_rating          0
rating_number           0
features                0
description             0
price               94886
images                  0
videos                  0
store               11331
categories              0
details                 0
parent_asin             0
bought_together    112590
dtype: int64


In [16]:
# Inspect a few sample records
for i in range(3):
    print(f"\n--- Product {i+1} ---")
    for key, value in meta_df.iloc[i].items():
        display_val = str(value)[:200] if isinstance(value, (str, list)) else value
        print(f"{key}: {display_val}")


--- Product 1 ---
main_category: All Beauty
title: Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack)
average_rating: 4.8
rating_number: 10
features: []
description: []
price: nan
images: [{'thumb': 'https://m.media-amazon.com/images/I/41qfjSfqNyL._SS40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41qfjSfqNyL.jpg', 'variant': 'MAIN', 'hi_res': None}
 {'thumb': 'https://m.media-amazon.com/images/I/41w2yznfuZL._SS40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41w2yznfuZL.jpg', 'variant': 'PT01', 'hi_res': 'https://m.media-amazon.com/images/I/71i77AuI9xL._SL1500_.jpg'}]
videos: []
store: Howard Products
categories: []
details: {'Package Dimensions': '7.1 x 5.5 x 3 inches; 2.38 Pounds', 'UPC': '617390882781'}
parent_asin: B01CUPMQZE
bought_together: None

--- Product 2 ---
main_category: All Beauty
title: Yes to Tomatoes Detoxifying Charcoal Cleanser (Pack of 2) with Charcoal Powder, Tomato Fruit Extract, and Gingko Biloba Leaf Extract, 5 fl. oz.
average_rating: 4.5
rati

## 2. Field Selection & Preprocessing

**Fields selected for retrieval:**
- `title` — primary product identifier
- `description` — detailed product information
- `features` — bullet-point product attributes

**Rationale:** These three fields contain the richest text for matching user queries. Price, rating, and images are kept as metadata for display but not included in the search text.

**Preprocessing decisions:**
- Concatenate title + description + features into a single `text` field per product
- Skip products with no text content
- For BM25: tokenize with lowercase + punctuation removal + whitespace split
- For semantic search: use raw concatenated text (model handles tokenization)

In [17]:
from src.utils import build_text

records = []
for _, row in meta_df.iterrows():
    product = row.to_dict()
    text = build_text(product)
    if not text.strip():
        continue
    records.append({
        "parent_asin": product.get("parent_asin", ""),
        "title": product.get("title", ""),
        "text": text,
        "price": product.get("price"),
        "average_rating": product.get("average_rating"),
    })

corpus_df = pd.DataFrame(records)
corpus_df.to_parquet(DATA_PROCESSED / "product_corpus.parquet", compression="zstd", index=False)
print(f"Corpus size: {len(corpus_df):,} products")
print("Saved to data/processed/product_corpus.parquet")
corpus_df.head()

Corpus size: 112,578 products
Saved to data/processed/product_corpus.parquet


,parent_asin,title,text,price,average_rating
0,B01CUPMQZE,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...","Howard LC0008 Leather Conditioner, 8-Ounce (4-...",NaN,4.8
1,B076WQZGPM,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,NaN,4.5
2,B000B658RI,Eye Patch Black Adult with Tie Band (6 Per Pack),Eye Patch Black Adult with Tie Band (6 Per Pack),NaN,4.4
3,B088FKY3VD,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...","Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",NaN,3.1
4,B07NGFDN6G,Precision Plunger Bars for Cartridge Grips – 9...,Precision Plunger Bars for Cartridge Grips – 9...,NaN,4.3


In [18]:
print("Price statistics:")
print(corpus_df["price"].describe())
print("\nRating statistics:")
print(corpus_df["average_rating"].describe())
print("\nText length statistics:")
corpus_df["text_len"] = corpus_df["text"].str.len()
print(corpus_df["text_len"].describe())

Price statistics:
count    17703.000000
mean        27.256962
std         50.473179
min          0.010000
25%          9.990000
50%         16.990000
75%         29.900000
max       2548.980000
Name: price, dtype: float64

Rating statistics:
count    112578.000000
mean          3.883458
std           0.874400
min           1.000000
25%           3.400000
50%           4.000000
75%           4.500000
max           5.000000
Name: average_rating, dtype: float64

Text length statistics:
count    112578.000000
mean        113.591608
std          53.178264
min           1.000000
25%          67.000000
50%         114.000000
75%         158.000000
max        1455.000000
Name: text_len, dtype: float64


## 3. BM25 Retriever

Building a keyword-based retrieval system using BM25 (Okapi BM25).

In [19]:
from src.bm25 import BM25Retriever
from src.utils import load_corpus

corpus = load_corpus(str(DATA_PROCESSED / "product_corpus.parquet"))

bm25 = BM25Retriever()
bm25.build_index(corpus)
bm25.save(str(INDICES_DIR / "bm25_index.pkl"))
print("BM25 index built and saved.")

results = bm25.search("vitamin C serum", top_k=5)
print("\nTest query: 'vitamin C serum'")
for i, r in enumerate(results):
    print(f"{i+1}. {r['title']} (score: {r['score']:.4f})")

BM25 index built and saved.

Test query: 'vitamin C serum'
1. Organic Vitamin C Serum for Face-Professional Strength-Organic Vitamin C Serum-Hyaluronic Acid-Vitamin E-Naturally Derived 20% Vitamin C The Best Vitamin C Serum (1 Ounce) (score: 21.4679)
2. 24K Vitamin C Serum for Face 2 PACK, Facial Serum with Vitamin C and Hyaluronic Acid, Facial Moisturizer 2 Pack Vitamin C Serum (score: 21.0783)
3. Renew Vitamin C Serum (score: 20.8527)
4. Vitamin C Serum 1oz (score: 20.8527)
5. Vitamin C Plus Serum (score: 20.8527)


**Observations on BM25 results for "vitamin C serum":**

- BM25 performs well on this query because the search terms appear verbatim in product titles (hence "keyword retrieval").
- The top two results score higher (~21.1–21.5) because they repeat "vitamin C serum" multiple times in the title, which BM25 rewards via term frequency.
- Results 3–5 all tie at ~20.85, because each contains the exact phrase once in a short title with little other text to dilute the score.
- A limitation is visible here which is that BM25 has no understanding of *meaning*. A query like "face brightening serum with antioxidants" would miss these same products even though vitamin C serums are exactly that. This is the gap we expect semantic retrieval to fill.

## 4. Semantic Retriever

Building a semantic search system using sentence-transformers (all-MiniLM-L6-v2) and FAISS.

In [ ]:
from src.semantic import SemanticRetriever

semantic = SemanticRetriever()
semantic.build_index(corpus)
semantic.save(str(INDICES_DIR / "faiss_index"))
print("Semantic index built and saved.")

results = semantic.search("vitamin C serum", top_k=5)
print("\nTest query: 'vitamin C serum'")
for i, r in enumerate(results):
    print(f"{i+1}. {r['title']} (score: {r['score']:.4f})")

print("\n---")
results2 = semantic.search("something to protect from sun damage", top_k=5)
print("\nTest query: 'something to protect from sun damage'")
for i, r in enumerate(results2):
    print(f"{i+1}. {r['title']} (score: {r['score']:.4f})")

print("\n---")
results3 = semantic.search("face brightening serum with antioxidants", top_k=5)
print("\nTest query: 'face brightening serum with antioxidants'")
for i, r in enumerate(results3):
    print(f"{i+1}. {r['title']} (score: {r['score']:.4f})")

/Users/williamchong/miniforge3/envs/575-project/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7824.91it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Semantic index built and saved.

Test query: 'vitamin C serum'
  1. PURE VITAMIN C SERUM (score: 0.9416)
  2. Vitamin C Plus Serum (score: 0.9233)
  3. Vitamin C Serum 1oz (score: 0.8743)
  4. Renew Vitamin C Serum (score: 0.8461)
  5. RA Herbals Ultimate Vitamin C Serum (score: 0.7833)

---

Test query: 'something to protect from sun damage'
  1. Near Skin Dustless Defense Sun Block (score: 0.6586)
  2. UV Capture PLUS Pure Mild Sun Cream Sun Block Sunscreen SPF PA +++ + Moistful Essential Sun Block Cream With Skin Relieving Protection Barrier (score: 0.5880)
  3. [The Saem] Eco Earth Power Sun Cream 50g (Waterproof Sun Block) (score: 0.5658)
  4. Banana Boat Sunscreen Simply Protect Kids Tear Free, Broad Spectrum Mineral Sunscreen Spray, TSA Approved Size, SPF 50+, 1.8 oz (score: 0.5587)
  5. Solarbiome Ampule Sunscreen Sun Block Sunscreen SPF + 50 PA +++ Dust Blocking UV Protector, 50ml (score: 0.5275)

---

Test query: 'face brightening serum with antioxidants'
  1. Majestic Pure S

**Observations on semantic retriever results:**

- **"vitamin C serum"** — Semantic search returns relevant results (all are vitamin C serums), but ranks them differently than BM25. "PURE VITAMIN C SERUM" scores highest (0.94) despite having a short title, because the embedding captures meaning rather than rewarding keyword repetition. BM25 favored longer titles that repeated the phrase multiple times.
- **"something to protect from sun damage"** — This is where semantic search shines. None of these results contain the exact words "protect from sun damage," yet the model correctly retrieves sunscreen/sun block products by understanding the *intent* behind the query. BM25 would struggle here because the query terms don't match product vocabulary (e.g., "SPF", "sun block", "sunscreen").
- **"face brightening serum with antioxidants"** — Semantic search retrieves a mix of brightening serums and antioxidant products, including a Vitamin C serum (result 5), which is contextually correct since vitamin C is an antioxidant. This demonstrates the model's ability to connect related concepts that don't share exact keywords.
- **Scores** range from ~0.5 to ~0.94 (cosine similarity). Exact keyword matches score highest; intent-based queries score lower but still retrieve relevant products. This is expected — the model has to "bridge" a larger semantic gap for paraphrased queries.
- **Limitation:** The model returned a duplicate (Trader Joe's Antioxidant Serum appears twice), which suggests duplicate or near-duplicate products exist in the corpus. Deduplication could improve result diversity.

## 5. Hybrid Retriever

Combining BM25 and semantic scores with weighted linear combination.

*Will completes this section.*

## 6. Qualitative Evaluation

Running all three methods on our evaluation query set and comparing results.

*Will completes this section.*